In [4]:
import os
from google.colab import userdata

os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
os.environ['TAVILY_API_KEY'] = userdata.get('TAVILY_API_KEY')

#loading the gemini_api_key & tavily_api_key from google colab into this notebook
gemini_api_key = userdata.get('GEMINI_API_KEY')
tavily_api_key = userdata.get('TAVILY_API_KEY')

print(f"Gemini API key: ",bool(gemini_api_key))
print(f"Tavily API key: ",bool(tavily_api_key))

Gemini API key:  True
Tavily API key:  True


In [5]:
#installing all the required libraries
!pip install -U langchain langchain-google-genai langgraph

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.2/250.2 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.5/263.5 kB 16.2 MB/s eta 0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled google-genai-2.12.1
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.2.11
    Uninstalling langgraph-1.2.11:
      Successfully uninstalled langgraph-1.2.11
  

In [17]:
#importing different types of messages
from langchain.messages import HumanMessage
from langchain.messages import AIMessage

In [9]:
#installing tavily python library(this is use for web search)
!pip install tavily-python

In [16]:
from langchain.tools import tool
from typing import Dict,Any
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=tavily_api_key)

@tool
def search_recipes(query: str) -> Dict[str,Any]:
  """Search the web for recipes from the ingredients mention in the query"""
  return tavily_client.search(query)

search_recipes.invoke("I have chicken , yogurt and masales in my kitchen What can I make from it?")

{'query': 'I have chicken , yogurt and masales in my kitchen What can I make from it?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://thenoshery.com/crispy-indian-spiced-yogurt-chicken',
   'title': 'Crispy Indian Spiced Yogurt Chicken - The Noshery',
   'content': 'Combine 7 oz. of greek yogurt with 3 1/2 tsp of Gram Masala, stir and taste to check if you like more seasoning. Add green onions, garlic and milk, stir till well combined.\n\nDredge chicken in yogurt mixture and then in panko bread crumbs.\n\nPlace chicken in buttered pan, place thin pads of butter on top of the chicken & cover with foil, bake at 350 degrees for 35 mins. Tranfer chicken to broiler set to “NORM”, toast panko bread crumbs until golden brown.\n\n### xoxo [...] Picnik collage\n\nCombine 7 oz. of greek yogurt with 3 1/2 tsp of Gram Masala, stir and taste to check if you like more seasoning. Add green onions, garlic and milk, stir till well combined.\n\nIndian Spice 

In [19]:
#creating a base model class for response
from pydantic import BaseModel

class Recipes_Info(BaseModel):
  ingredients : str
  cooking_process : str

In [28]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model = "gemini-3.5-flash-lite",
    temperature = 0.5,
    api_key = gemini_api_key,
    system_prompt = "You are a master chef, You can make dish from the given ingredients. You can use the tool from web search for recipes and return the ingredients and coooking process required to make the dish "
)

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3553: UserWarning: WARNING! system_prompt is not default parameter.
                system_prompt was transferred to model_kwargs.
                Please confirm that system_prompt is what you intended.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [30]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
agent = create_agent(
    model = model,
    tools = [search_recipes],
    checkpointer = InMemorySaver(),
    response_format = Recipes_Info
)

In [33]:
from pprint import pprint

query = HumanMessage(content = "I have chicken , yogurt and some indian masala in my kitchen , can you suggest and indian dish")
config = {"configurable":{"thread_id":"1"}}

response = agent.invoke({"messages":[query]},config=config)
pprint(response)
print("------------------------------------------------------------------------------------------------------------------------------------")
print(response["structured_response"].ingredients)
print(response["structured_response"].cooking_process)
print("------------------------------------------------------------------------------------------------------------------------------------")

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


{'messages': [HumanMessage(content='I have chicken , yogurt and some indian masala in my kitchen , can you suggest and indian dish', additional_kwargs={}, response_metadata={}, id='ce855adc-1d86-4862-8f91-ea2d2b3bccd8'),
              AIMessage(content=[{'type': 'text', 'text': '{\n  "ingredients": "chicken, yogurt, Indian masala, ginger, garlic, onion, tomatoes, oil or butter",\n  "cooking_process": "Marinate the chicken pieces in a mixture of yogurt and Indian masala (such as garam masala, turmeric, cumin, and chili powder) for at least 30 minutes. In a pan, sauté finely chopped onions, ginger, and garlic until golden brown. Add pureed tomatoes and cook until the oil separates. Add the marinated chicken to the pan and cook on medium heat until the chicken is tender and fully cooked. Garnish with fresh cilantro and serve hot with rice or naan."\n}', 'extras': {'signature': 'EmAKXgFpFH0TI7d5BRAuvYjpTzEX9/fO5ADKtoDJ3cOcEnajjeiO1KSqG23PMqSCoHse04/djy23Y658iGRNYUz8zOZ0gBo+sJ7Zsl079Ymazf1s

In [39]:
query = HumanMessage(content="What was the name of the last recipe you have suggested?")

response = agent.invoke({"messages":[query]}, config = config)
pprint(response['messages'])
pprint(response['messages'][-1])

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[HumanMessage(content='I have chicken , yogurt and some indian masala in my kitchen , can you suggest and indian dish', additional_kwargs={}, response_metadata={}, id='ce855adc-1d86-4862-8f91-ea2d2b3bccd8'),
 AIMessage(content=[{'type': 'text', 'text': '{\n  "ingredients": "chicken, yogurt, Indian masala, ginger, garlic, onion, tomatoes, oil or butter",\n  "cooking_process": "Marinate the chicken pieces in a mixture of yogurt and Indian masala (such as garam masala, turmeric, cumin, and chili powder) for at least 30 minutes. In a pan, sauté finely chopped onions, ginger, and garlic until golden brown. Add pureed tomatoes and cook until the oil separates. Add the marinated chicken to the pan and cook on medium heat until the chicken is tender and fully cooked. Garnish with fresh cilantro and serve hot with rice or naan."\n}', 'extras': {'signature': 'EmAKXgFpFH0TI7d5BRAuvYjpTzEX9/fO5ADKtoDJ3cOcEnajjeiO1KSqG23PMqSCoHse04/djy23Y658iGRNYUz8zOZ0gBo+sJ7Zsl079Ymazf1spaQGSBTzIB1MAb7NBIQ='}}], 